In [ ]:
# ==============================================================================
# 📓 Advanced RAG Cookbook: 04_advanced_retrieval_and_reranking.ipynb
# ==============================================================================
# الهدف: إتقان تقنيات الاسترجاع المتقدمة في الشركات (Production Retrieval):
# 1. Dense vs Sparse Retrieval (Vector Search vs BM25 Keyword Search)
# 2. Hybrid Search (Ensemble Retriever مع RRF Scoring)
# 3. Re-ranking Engine (استخدام Cross-Encoders لإعادة ترتيب النتائج بحرفية)
# ==============================================================================

# تثبيت المكتبات المطلوبة للتشغيل
# !pip install langchain-community langchain-chroma rank_bm25 sentence-transformers flashrank

import os
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import Qdrant
from langchain.retrievers import EnsembleRetriever
from langchain_core.documents import Document
from sentence_transformers import CrossEncoder

print("✅ تم استيراد كل مكتبات الاسترجاع المتقدم والـ Reranking بنجاح!")

<div dir="rtl">

## 1. لماذا يحتاج الـ Production RAG إلى Hybrid Search & Re-ranking؟

### 📌 المشكلة في الـ Semantic Vector Search العادي (Dense Only):
البحث بالـ Embeddings يفهم **المعنى والسياق** ممتاز جداً، ولكنه يفشل في حالات مهمة جداً داخل الشركات:
* **الأرقام والأكواد:** زي البحث عن رقم فاتورة `INV-2026-99` أو كود خطأ `Error 404`.
* **أسماء المنتجات والرموز:** زي `GPT-4o` أو أسماء الأدوية والمصطلحات الطبية والتقنية.

### 💡 الحل الشامل في الشركات (Hybrid Search + Re-ranking Pipeline):

```
سؤال المستخدم (User Query)
   │
   ├───► 1. Dense Retriever (Vector Search / Cosine)  ──► يجيب أسرع 20 قطعة بناءً على المعنى
   │
   └───► 2. Sparse Retriever (BM25 / Keyword Match)   ──► يجيب أسرع 20 قطعة بناءً على الكلمات المفتاحية
                               │
                               ▼
                 3. Ensemble (Reciprocal Rank Fusion - RRF)
                  (دمج الـ 40 قطعة وإزالة التكرار)
                               │
                               ▼
                 4. Re-ranker Model (Cross-Encoder)
                  (إعادة حساب أثر ونسبة مطابقة كل قطعة بدقة متناهية)
                               │
                               ▼
                تصفية أفضل Top-3 Chunks وتمريرها للـ LLM
```

</div>

In [ ]:
# ==============================================================================
# 2. تجهيز البيانات التجريبية للمقارنة
# ==============================================================================

# بيانات تجمع بين أرقام دقيقة، أكواد، ومفاهيم عامة
documents_dataset = [
    Document(page_content="Error code ERR-809: Database connection timeout during peak traffic."),
    Document(page_content="To resolve database connection issues, restart the primary cluster node."),
    Document(page_content="Project Titan update: Client delivery date is scheduled for Q4 2026."),
    Document(page_content="Our system handles high concurrency using asynchronous Python FastAPI services."),
    Document(page_content="For billing issues with invoice INV-9042, contact finance@company.com.")
]

# 1. إعداد الـ Vector Store (Dense Retriever)
embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_db = Qdrant.from_documents(documents_dataset, embeddings_model, location=":memory:")
dense_retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# 2. إعداد الـ BM25 Keyword Search (Sparse Retriever)
sparse_retriever = BM25Retriever.from_documents(documents_dataset)
sparse_retriever.k = 3

print("✅ تم إعداد الـ Dense والـ Sparse Retrievers بنجاح!")

<div dir="rtl">

## 2. مقارنة سريعة: Dense Search vs Sparse Search

سنلاحظ الآن كيف يفشل الـ Vector Search عند البحث عن كود دقيق بينما ينجح الـ BM25 والعكس!

</div>

In [ ]:
# ==============================================================================
# 3. Test Dense vs Sparse Search
# ==============================================================================

query = "How do I fix ERR-809?"

print(f"🔍 Query: {query}\n")

print("--- 1. Dense Retriever (Vector Search) Results ---")
dense_results = dense_retriever.invoke(query)
for doc in dense_results:
    print(f"• {doc.page_content}")

print("--- 2. Sparse Retriever (BM25 Keyword Search) Results ---")
sparse_results = sparse_retriever.invoke(query)
for doc in sparse_results:
    print(f"• {doc.page_content}")

<div dir="rtl">

## 3. تطبيق الـ Hybrid Search (Ensemble Retriever + RRF)

يقوم الـ **EnsembleRetriever** بالجمع بين نتائج الـ BM25 والـ Vector Search وتطوير ترتيب هجين باستخدام خوارزمية **Reciprocal Rank Fusion (RRF)** مع إمكانية إعطاء أوزان (`weights`) لكل مسترجع.

</div>

In [ ]:
# ==============================================================================
# 4. Hybrid Search Setup (EnsembleRetriever)
# ==============================================================================

# إعطاء وزنين: 50% للـ Vector و 50% للـ BM25
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.5, 0.5]
)

hybrid_results = hybrid_retriever.invoke(query)

print(f"--- Hybrid Search Results for: '{query}' ---")
for i, doc in enumerate(hybrid_results):
    print(f"[{i+1}] {doc.page_content}")

<div dir="rtl">

## 4. إضافة خوارزمية الـ Re-ranking (Cross-Encoder Engine)

### 📌 الفرق الجوهري بين Bi-Encoder و Cross-Encoder:
* **Bi-Encoder (الـ Vector DB العادي):** يحول النصين لـ Vectors بشكل منفصل ثم يقيس المسافة. سريع جداً لكنه يفتقد التفاعل الدقيق بين الكلمات.
* **Cross-Encoder (الـ Re-ranker):** يمرر السؤال والقطعة **سوياً** داخل النموذج في نفس الوقت لحساب درجة المطابقة الدلالية بأعلى دقة ممكنة (High Precision)، ولكنه أبطأ، لذلك نطبقه فقط على أسرع 10 إلى 20 قطعة مسترجعة.

</div>

In [ ]:
# ==============================================================================
# 5. Re-ranking Pipeline with Cross-Encoder
# ==============================================================================

# 1. تحميل نموذج Cross-Encoder مخصص للـ Re-ranking
reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_documents(query, documents, top_n=2):
    """دالة إعادة الترتيب للـ Chunks بناءً على الـ Cross-Encoder Score"""
    # تجهيز أزواج (Query, Chunk_Content)
    pairs = [[query, doc.page_content] for doc in documents]
    
    # حساب الـ Scores المتطورة
    scores = reranker_model.predict(pairs)
    
    # دمج الـ Docs مع أرقام النقاط وترتيبهم تنازلياً
    doc_score_pairs = list(zip(documents, scores))
    ranked_pairs = sorted(doc_score_pairs, key=lambda x: x[1], reverse=True)
    
    # ارجاع أفضل Top N
    return ranked_pairs[:top_n]

# تطبيق الـ Re-ranking على نتائج الـ Hybrid Search
top_reranked = rerank_documents(query, hybrid_results, top_n=2)

print(f"--- Final Re-ranked Results for: '{query}' ---")
for doc, score in top_reranked:
    print(f"🎯 Score: {score:.4f} | Content: {doc.page_content}")

<div dir="rtl">

## 📝 الخلاصة والتوصيات للـ Production RAG

1. **لا تعتمد على Vector Search فقط:** في الشركات، استخدم دائماً **Hybrid Search** بدمج الـ BM25 مع الـ Vector DB لحماية نظامك من الفشل أمام المصطلحات والأكواد.
2. **استراتيجية التكاليف والأداء:** جلب عدد أكبر من القطع (مثلاً `k=20`) عبر الـ Hybrid Search، ثم مررهم على **Cross-Encoder Re-ranker** لفلترتهم لأفضل `top_k=3` فقط قبل تمريرهم للـ LLM.
3. **توفير التكلفة والـ Latency:** هذه الاستراتيجية توفر تكلفة الـ LLM Tokens بشكل هائل وتمنع الهلاوس (Hallucinations) لأنها تضمن إرسال أدق قطع ممكنة.

</div>